<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/EXP_Infotrade_Exportaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests pandas numpy ipywidgets google-api-python-client --quiet
print("✅ Librerías listas")
import os, io, json, time, random, unicodedata, calendar
import requests
import numpy as np
import pandas as pd
import ipywidgets as w
from datetime import date
from IPython.display import display

In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
# Regiones: selección múltiple (Ctrl/Cmd + clic). Sector y Subsector: una opción a la vez.
REGIONES = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Callao",
            "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín", "La Libertad", "Lambayeque",
            "Lima Metropolitana", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura",
            "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"]
SECTORES = ["Todos", "No Tradicional", "Tradicional", "Sin clasificación"]
SUBSECTORES = ["Todos", "AGROPECUARIO", "ARTESANÍAS", "MADERAS Y PAPELES", "METAL-MECÁNICO",
               "MINERÍA NO METÁLICA", "MINEROS", "PESQUERO", "PETRÓLEO Y GAS NATURAL",
               "PIELES Y CUEROS", "QUÍMICO", "SIDERO-METALÚRGICO", "SIN CLASIFICACIÓN",
               "TEXTIL", "VARIOS (inc. joyería)"]
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Setiembre", "Octubre", "Noviembre", "Diciembre"]

_hoy = date.today()
_disp = (_hoy.year * 12 + _hoy.month - 1) - 3          # por defecto: hoy menos 3 meses
_años = list(range(_hoy.year, 1993, -1))                 # la web ofrece 1994 en adelante
_mes = [(m, i + 1) for i, m in enumerate(MESES)]
_ancho = {"description_width": "150px"}

sel_anio_ini = w.Dropdown(options=_años, value=2005, description="Año inicio", style=_ancho)
sel_mes_ini = w.Dropdown(options=_mes, value=1, description="Mes inicio", style=_ancho)
sel_anio_disp = w.Dropdown(options=_años, value=_disp // 12, description="Disponible hasta (año)", style=_ancho)
sel_mes_disp = w.Dropdown(options=_mes, value=_disp % 12 + 1, description="Disponible hasta (mes)", style=_ancho)
sel_regiones = w.SelectMultiple(options=REGIONES, value=tuple(REGIONES), rows=13,
                                description="Regiones", style=_ancho,
                                layout=w.Layout(width="420px"))
btn_todas = w.Button(description="Marcar todas", layout=w.Layout(width="130px"))
btn_ninguna = w.Button(description="Desmarcar todas", layout=w.Layout(width="130px"))
btn_todas.on_click(lambda _: setattr(sel_regiones, "value", tuple(REGIONES)))
btn_ninguna.on_click(lambda _: setattr(sel_regiones, "value", ()))
sel_sector = w.Dropdown(options=SECTORES, value="Todos", description="Sector", style=_ancho)
sel_subsector = w.Dropdown(options=SUBSECTORES, value="Todos", description="Subsector", style=_ancho,
                           layout=w.Layout(width="420px"))
txt_partida = w.Text(value="", placeholder="vacío = todas (p. ej. 4418990000)", description="Partida", style=_ancho)
txt_empresa = w.Text(value="", placeholder="vacío = todas", description="Empresa", style=_ancho)
txt_usuario = w.Text(value="", placeholder="opcional: tu usuario de Infotrade", description="Usuario (auditoría)", style=_ancho)

display(w.VBox([
    w.HTML("<b>Período</b> (se descarga en cortes de 6 meses: Ene–Jun y Jul–Dic). "
           "En «Disponible hasta» pon el mes que indica la web: «Datos disponibles hasta …»."),
    w.HBox([sel_anio_ini, sel_mes_ini]),
    w.HBox([sel_anio_disp, sel_mes_disp]),
    w.HTML("<b>Regiones</b> (Ctrl/Cmd + clic para elegir varias)"),
    w.HBox([sel_regiones, w.VBox([btn_todas, btn_ninguna])]),
    w.HTML("<b>Filtros</b> (Sector y Subsector: una opción a la vez)"),
    sel_sector, sel_subsector, txt_partida, txt_empresa, txt_usuario,
]))

In [ ]:
# ═══ BLOQUE 3 · DESCARGA, CONTROL Y DRIVE ═══
# Usa lo elegido en el Bloque 2. Si se corta (Colab se desconecta), vuelve a ejecutarlo:
# salta los cortes que ya están en Drive con control OK y sigue donde quedó.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1JJqcvUomccL-dCCNm5yovl-9TN5laKYi"           # CSV de datos
DRIVE_FOLDER_CONTROL_ID = "1gAO8-UjTV2bmBwMPgONBAYeVfpwXSWCY"   # CONTROL y LOG
PAUSA = (2.0, 4.0)             # segundos aleatorios entre cortes (cortesía con el servidor)
TIMEOUT_EXCEL = 600            # segundos: la descarga de un corte grande puede tardar

API = "https://infotrade.promperu.gob.pe/wss/api"
PAGINA = "https://infotrade.promperu.gob.pe/reporte-exportaciones"
ruta_guardado = "/content/Exportaciones_Infotrade"
os.makedirs(ruta_guardado, exist_ok=True)

# ── 2. LEER LOS SELECTORES ───────────────────────────────────────────────────
inicio = (sel_anio_ini.value, sel_mes_ini.value)
fin = (sel_anio_disp.value, sel_mes_disp.value)
regiones = list(sel_regiones.value)
SECTOR, SUBSECTOR = sel_sector.value, sel_subsector.value
PARTIDA, EMPRESA = txt_partida.value.strip(), txt_empresa.value.strip()
USUARIO = txt_usuario.value.strip()
if inicio > fin:
    raise ValueError(f"El inicio {inicio} es posterior a «Disponible hasta» {fin}.")
if not regiones:
    raise ValueError("Elige al menos una región en el Bloque 2.")

def cortes(ini, fin):
    """Semestres (Ene–Jun / Jul–Dic) recortados al rango [ini, fin]. -> (año, sem, (a,m), (a,m), completo)"""
    out = []
    for a in range(ini[0], fin[0] + 1):
        for s, (m1, m2) in ((1, (1, 6)), (2, (7, 12))):
            d, h = max((a, m1), ini), min((a, m2), fin)
            if d <= h:
                out.append((a, s, d, h, (d, h) == ((a, m1), (a, m2))))
    return out

CORTES = cortes(inicio, fin)
ETIQ_FILTRO = " - ".join(x for x in [
    f"Sector {SECTOR}", f"Subsector {SUBSECTOR}",
    f"Partida {PARTIDA}" if PARTIDA else "", f"Empresa {EMPRESA}" if EMPRESA else ""] if x)
print(f"Período: {MESES[inicio[1]-1]} {inicio[0]} – {MESES[fin[1]-1]} {fin[0]}  →  {len(CORTES)} corte(s)")
print(f"Regiones: {len(regiones)}  |  {ETIQ_FILTRO}")
print(f"Descargas posibles: {len(CORTES) * len(regiones)}\n")

def nombre_dato(año, sem, region):
    return f"EXP {año}-S{sem} - {region} - {ETIQ_FILTRO}.csv"

NOMBRE_CONTROL = f"CONTROL Exportaciones - {ETIQ_FILTRO}.csv"

# ── 3. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)

for fid in (DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID):
    try:
        c = drive.files().get(fileId=fid, fields="id,name,mimeType,capabilities(canAddChildren)",
                              supportsAllDrives=True).execute()
    except HttpError as e:
        raise PermissionError(f"Sin acceso a la carpeta de Drive {fid} (HTTP {e.resp.status}).") from e
    if c.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError(f"El ID {fid} no es una carpeta de Drive.")
    if not c.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{c['name']}'.")
    print(f"☁️  Drive conectado -> carpeta: '{c['name']}'")

def archivos_en(folder_id):
    """{nombre: id} de los archivos de una carpeta (con paginación)."""
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def subir(ruta_local, folder_id, existentes, reintentos=5):
    nombre = os.path.basename(ruta_local)
    for i in range(1, reintentos + 1):
        try:
            media = MediaFileUpload(ruta_local, mimetype="text/csv", resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media,
                                     supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/{reintentos}: {e}")
            time.sleep(2 ** i)
    print(f"   ❌ No se pudo subir {nombre}. Queda en {ruta_local}")
    return False

def leer_csv_drive(file_id):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
    hecho = False
    while not hecho:
        _, hecho = dl.next_chunk()
    buf.seek(0)
    return pd.read_csv(buf, dtype=str, encoding="utf-8-sig")

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)

# Control acumulado de corridas anteriores (sirve para reanudar)
COLS_CONTROL = ["AÑO", "SEMESTRE", "DESDE", "HASTA", "CORTE_COMPLETO", "REGION", "SECTOR", "SUBSECTOR",
                "PARTIDA", "EMPRESA", "FILAS_DESCARGADAS", "FILAS_REF", "DIF_FILAS",
                "FOB_USD_DESCARGADO", "FOB_USD_REF", "DIF_FOB_USD",
                "PESO_NETO_KG_DESCARGADO", "PESO_NETO_KG_REF", "DIF_PESO_NETO_KG",
                "PESO_BRUTO_KG_DESCARGADO", "PESO_BRUTO_KG_REF", "DIF_PESO_BRUTO_KG",
                "CANTIDAD_DESCARGADA", "CANTIDAD_REF", "DIF_CANTIDAD",
                "REGIONES_EN_DATOS", "ESTADO", "ARCHIVO", "FECHA_EXTRACCION"]
if NOMBRE_CONTROL in control_drive:
    control = leer_csv_drive(control_drive[NOMBRE_CONTROL])
    print(f"📄 Control previo encontrado: {len(control)} corte(s) registrados")
else:
    control = pd.DataFrame(columns=COLS_CONTROL)

# ── 4. API DE INFOTRADE ──────────────────────────────────────────────────────
sesion = requests.Session()
sesion.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json", "Content-Type": "application/json",
    "Origin": "https://infotrade.promperu.gob.pe", "Referer": PAGINA})

def llamar(ruta, cuerpo, timeout=60, reintentos=4):
    """POST JSON con reintentos y backoff. -> (json, error)."""
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.post(f"{API}/{ruta}", json=cuerpo, timeout=timeout)
            if r.status_code == 200:
                return r.json(), None
            ultimo = f"HTTP {r.status_code}"
        except (requests.RequestException, ValueError) as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(60, 3 * 2 ** i + random.random()))
    return None, f"{ruta}: {ultimo}"

def filtros(d, h, region):
    ult = calendar.monthrange(*h)[1]
    return {"FechaInicio": f"01-{d[1]:02d}-{d[0]}", "FechaFin": f"{ult:02d}-{h[1]:02d}-{h[0]}",
            "Empresa": EMPRESA, "Mercado": "", "Partida": PARTIDA, "Departamento": region,
            "Tipo": "" if SECTOR == "Todos" else SECTOR,
            "Sector": "" if SUBSECTOR == "Todos" else SUBSECTOR,
            "DescripcionComercial": "", "DescripcionProducto1": "", "Condicion2": "",
            "DescripcionProducto2": "", "Condicion3": "", "DescripcionProducto3": "",
            "DescripcionProducto1E": "", "Condicion2E": "", "DescripcionProducto2E": "",
            "Condicion3E": "", "DescripcionProducto3E": ""}

def auditoria(f, evento):
    if not USUARIO:
        return
    j = dict(f, Boton=evento, Navegador="Chrome", Dispositivo="Windows", Ip="")
    llamar("user/RegistrarAuditoria", {
        "UsuarioAuditoria": USUARIO, "CodigoOpcionAuditoria": "Reportes",
        "CodigoPaginaAuditoria": "Exportaciones", "CodigoSubPaginaAuditoria": "",
        "Evento": evento, "Json": json.dumps(j, ensure_ascii=False),
        "Pais": "Todos", "Partida": f["Partida"], "Ruc": ""}, reintentos=1)

COLUMNAS = {  # API -> CSV (nombres como en la web)
    "FechaEmbarque": "FECHA_EMBARQUE", "Ruc": "RUC", "RazonSocial": "RAZON_SOCIAL",
    "Departamento": "REGION", "CodigoPartida": "PARTIDA", "Partida": "DESCRIPCION_ARANCELARIA",
    "DescripcionComercial": "DESCRIPCION_COMERCIAL", "PaisDestino": "MERCADO_DESTINO",
    "Monto": "MONTO_FOB_USD", "PesoNeto": "PESO_NETO_KG", "PesoBruto": "PESO_BRUTO_KG",
    "Cantidad": "CANTIDAD", "TipoSector": "SECTOR", "Sector": "SUBSECTOR"}
NUMERICAS = {"Monto": "FOB_USD", "PesoNeto": "PESO_NETO_KG", "PesoBruto": "PESO_BRUTO_KG", "Cantidad": "CANTIDAD"}

def a_num(x):
    try:
        return float(str(x).replace(",", "")) if x not in (None, "") else 0.0
    except ValueError:
        return np.nan

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t or ""))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

def descargar_corte(año, sem, d, h, completo, region):
    """Secuencia de la web: concurrencia -> consulta -> Excel -> concurrencia (1, 2) -> auditoría."""
    f = filtros(d, h, region)
    llamar("Comunes/ObtenerParametroConcurrenciaReporteActual", {"headers": {"Accept": "application/json"}})
    ref, err = llamar("Reportes/Exportaciones", dict(f, NumeroPagina=1))
    if err:
        return None, None, err
    total = int(ref.get("TotalRegistros") or 0)
    tot = ref.get("Totales") or {}
    filas = []
    if total > 0:
        exc, err = llamar("Reportes/ExportacionesExcel", f, timeout=TIMEOUT_EXCEL, reintentos=3)
        if err:
            return None, None, err
        filas = exc.get("RptData") or []
        llamar("Comunes/ActualizarConcurrenciaDescarga", {"Tipo": 1, "PerfilUsuario": "PERFIL_USUARIO_EXTERNO"}, reintentos=2)
        llamar("Comunes/ActualizarConcurrenciaDescarga", {"Tipo": 2, "PerfilUsuario": "PERFIL_USUARIO_EXTERNO"}, reintentos=2)
        auditoria(f, "EXPORTAR EXCEL")
    df = pd.DataFrame(filas)
    for c in COLUMNAS:
        if c not in df.columns:
            df[c] = ""
    df = df[list(COLUMNAS)]
    for c in NUMERICAS:
        df[c] = pd.to_numeric(df[c].map(a_num), errors="coerce").fillna(0.0)
    for c in ("Ruc", "CodigoPartida"):          # texto: conservar ceros a la izquierda
        df[c] = df[c].astype(str).replace({"None": "", "nan": ""})

    # Control del corte: filas y totales vs la consulta de la web
    fila = {"AÑO": año, "SEMESTRE": f"S{sem}", "DESDE": f["FechaInicio"], "HASTA": f["FechaFin"],
            "CORTE_COMPLETO": "SI" if completo else "NO", "REGION": region, "SECTOR": SECTOR,
            "SUBSECTOR": SUBSECTOR, "PARTIDA": PARTIDA, "EMPRESA": EMPRESA,
            "FILAS_DESCARGADAS": len(df), "FILAS_REF": total, "DIF_FILAS": len(df) - total}
    estado = "OK" if len(df) == total else "REVISAR"
    for api, col in NUMERICAS.items():
        desc = float(df[api].sum()) if len(df) else 0.0
        refv = a_num(tot.get(api)) if total else 0.0
        suf = "DESCARGADA" if col == "CANTIDAD" else "DESCARGADO"
        fila[f"{col}_{suf}"], fila[f"{col}_REF"] = round(desc, 3), refv
        fila[f"DIF_{col}"] = round(desc - refv, 3)
        if abs(desc - refv) > max(1.0, abs(refv) * 1e-9):
            estado = "REVISAR"
    regiones_datos = sorted(set(df["Departamento"].astype(str)))
    fila["REGIONES_EN_DATOS"] = "; ".join(regiones_datos)
    pedida = normalizar(region)
    if any(not (normalizar(r) in pedida or pedida in normalizar(r)) for r in regiones_datos):
        estado = "REVISAR"
    fila["ESTADO"] = estado
    fila["FECHA_EXTRACCION"] = date.today().isoformat()
    return df.rename(columns=COLUMNAS), fila, None

# ── 5. BUCLE: CORTE x REGIÓN (con reanudación) ───────────────────────────────
sesion.get(PAGINA, timeout=60)                 # cookies del sitio, como el navegador
hechos = set()
if len(control):
    ok = control[(control["ESTADO"] == "OK") & (control["CORTE_COMPLETO"] == "SI")]
    hechos = {(str(r["AÑO"]), r["SEMESTRE"], r["REGION"]) for _, r in ok.iterrows()
              if r.get("ARCHIVO") in datos_drive}

def guardar_control():
    ruta = os.path.join(ruta_guardado, NOMBRE_CONTROL)
    control[COLS_CONTROL].to_csv(ruta, index=False, encoding="utf-8-sig")
    subir(ruta, DRIVE_FOLDER_CONTROL_ID, control_drive)

tareas = [(c, r) for c in CORTES for r in regiones]
pendientes = [(c, r) for c, r in tareas if (str(c[0]), f"S{c[1]}", r) not in hechos]
print(f"Ya descargados con control OK: {len(tareas) - len(pendientes)}  |  pendientes: {len(pendientes)}\n")

errores, resumen = [], []
for n, ((año, sem, d, h, completo), region) in enumerate(pendientes, 1):
    etiqueta = f"[{n}/{len(pendientes)}] {año}-S{sem} {region}"
    df, fila, err = descargar_corte(año, sem, d, h, completo, region)
    if err:
        print(f"❌ {etiqueta}: {err}")
        errores.append({"AÑO": año, "SEMESTRE": f"S{sem}", "REGION": region, "ERROR": err})
        time.sleep(random.uniform(*PAUSA))
        continue
    nombre = nombre_dato(año, sem, region)
    ruta = os.path.join(ruta_guardado, nombre)
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    subio = subir(ruta, DRIVE_FOLDER_ID, datos_drive)
    os.remove(ruta)                                # no llenar el disco de Colab
    fila["ARCHIVO"] = nombre if subio else ""
    clave = (control["AÑO"].astype(str) == str(año)) & (control["SEMESTRE"] == f"S{sem}") & (control["REGION"] == region)
    control = pd.concat([control[~clave], pd.DataFrame([fila])], ignore_index=True)
    guardar_control()                              # se guarda tras cada corte: permite reanudar
    marca = "✅" if fila["ESTADO"] == "OK" else "⚠️"
    print(f"{marca} {etiqueta}: {fila['FILAS_DESCARGADAS']:,} filas | FOB {fila['FOB_USD_DESCARGADO']:,.2f} USD | {fila['ESTADO']}")
    resumen.append(fila)
    time.sleep(random.uniform(*PAUSA))

# ── 6. RESUMEN ───────────────────────────────────────────────────────────────
print("═" * 70)
print(f"Descargados en esta corrida: {len(resumen)}  |  con error: {len(errores)}")
if resumen:
    display(pd.DataFrame(resumen)["ESTADO"].value_counts().to_frame("cortes"))
    rev = pd.DataFrame(resumen)
    rev = rev[rev["ESTADO"] != "OK"]
    if len(rev):
        print("⚠️ Cortes a revisar (no cuadran con la consulta de la web):")
        display(rev[["AÑO", "SEMESTRE", "REGION", "DIF_FILAS", "DIF_FOB_USD", "REGIONES_EN_DATOS"]])
if errores:
    print("❌ Cortes con error (vuelve a ejecutar el Bloque 3 para reintentarlos):")
    display(pd.DataFrame(errores))
    ruta_log = os.path.join(ruta_guardado, f"LOG Exportaciones {date.today().isoformat()} - {ETIQ_FILTRO}.csv")
    pd.DataFrame(errores).to_csv(ruta_log, index=False, encoding="utf-8-sig")
    subir(ruta_log, DRIVE_FOLDER_CONTROL_ID, control_drive)
faltan = len(pendientes) - len(resumen)
print("✅ Todo descargado." if not faltan else f"ℹ️ Faltan {faltan} corte(s): ejecuta de nuevo el Bloque 3.")
print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")